# Mapping Surface Urban Heat Island Change in Greater Concepción, Chile (2015-2026)

This notebook implements a reproducible **Google Earth Engine + Python** workflow to map changes in **Surface Urban Heat Island (SUHI)** intensity in Greater Concepción, Chile, between summer 2015 and summer 2026.

The study area includes the municipalities of **Concepción, Talcahuano, Hualpén, and San Pedro de la Paz**. The analysis uses Landsat 8 Collection 2 Level 2 imagery, QA_PIXEL cloud masking, land surface temperature (LST), NDVI, MNDWI, NDBI, a spectral-index urban mask, and a spectral-index rural reference mask.

SUHI is calculated as each urban LST pixel minus the mean rural reference LST for the same summer period. The result is a relative land-surface thermal signal, not an air-temperature measurement.


## 1. Install dependencies

This cell detects whether the notebook is running in Google Colab. In Colab, it installs missing packages. In a local Python environment, it does not install or reinstall packages.


In [ ]:
import importlib
import subprocess
import sys


def running_in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


IN_COLAB = running_in_colab()

PACKAGE_IMPORTS = {
    "earthengine-api": "ee",
    "geemap": "geemap",
    "geopandas": "geopandas",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "rasterio": "rasterio",
    "shapely": "shapely",
}

if IN_COLAB:
    missing_packages = []
    for package_name, import_name in PACKAGE_IMPORTS.items():
        try:
            importlib.import_module(import_name)
        except ImportError:
            missing_packages.append(package_name)

    if missing_packages:
        print("Installing missing packages:", ", ".join(missing_packages))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
    else:
        print("All required packages are already available in Colab.")
else:
    print("Local environment detected. Install requirements.txt before running if packages are missing.")


## 2. Imports and output folders

This section imports the required packages and creates the output folders used by the workflow.


In [ ]:
from pathlib import Path
import warnings

import ee
import geemap
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from IPython.display import display
from matplotlib.colors import TwoSlopeNorm

warnings.filterwarnings("ignore")

OUTPUT_DIR = Path("outputs")
FIGURE_DIR = OUTPUT_DIR / "figures"
GEOTIFF_DIR = OUTPUT_DIR / "geotiffs"
TABLE_DIR = OUTPUT_DIR / "tables"

for folder in [FIGURE_DIR, GEOTIFF_DIR, TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Output folders ready.")


## 3. Initialize Google Earth Engine

Authenticate and initialize Earth Engine. The notebook supports both Google Colab and local Jupyter environments.


In [ ]:
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

print("Earth Engine initialized successfully.")


## 4. Load study area

The study area boundary is stored as a local shapefile at `data/admin_boundaries/GreaterConcepcion.shp`. The file is derived from IDE Chile DPA 2023 and represents Concepción, Talcahuano, Hualpén, and San Pedro de la Paz.


In [ ]:
AOI_PATH = Path("data/admin_boundaries/GreaterConcepcion.shp")
EXPECTED_MUNICIPALITIES = ["Concepción", "Talcahuano", "Hualpén", "San Pedro de la Paz"]

if not AOI_PATH.exists():
    raise FileNotFoundError(
        f"AOI shapefile not found: {AOI_PATH}. "
        "Place GreaterConcepcion.shp and its companion files in data/admin_boundaries/."
    )

aoi_gdf = gpd.read_file(AOI_PATH)

if aoi_gdf.crs is None:
    raise ValueError("The AOI shapefile has no CRS. Define the CRS before running this notebook.")

aoi_gdf = aoi_gdf.to_crs("EPSG:4326")

metadata_columns = [col for col in ["COMUNA", "NAME", "PROVINCIA", "REGION"] if col in aoi_gdf.columns]
print("Municipalities represented in the study area:")
for municipality in EXPECTED_MUNICIPALITIES:
    print(f"- {municipality}")

if metadata_columns:
    display(aoi_gdf[metadata_columns].drop_duplicates())
else:
    display(aoi_gdf.drop(columns="geometry", errors="ignore").head())

print(f"Number of AOI features: {len(aoi_gdf)}")
print(f"CRS: {aoi_gdf.crs}")

AOI_FC = geemap.geopandas_to_ee(aoi_gdf)
AOI_GEOM = AOI_FC.geometry()

Map = geemap.Map()
Map.centerObject(AOI_FC, 11)
Map.addLayer(AOI_FC, {"color": "red"}, "Greater Concepción study area")
Map


## 5. Landsat 8 preprocessing functions

The functions below apply QA_PIXEL cloud masking, Landsat Collection 2 Level 2 scale factors, LST conversion to degrees Celsius, spectral index calculation, and summer composite generation.


In [ ]:
LANDSAT_COLLECTION = "LANDSAT/LC08/C02/T1_L2"
MAX_CLOUD_COVER = 60
EXPORT_SCALE = 30

SUMMER_2015_START = "2015-01-01"
SUMMER_2015_END = "2015-03-31"
SUMMER_2026_START = "2026-01-01"
SUMMER_2026_END = "2026-03-31"

URBAN_NDVI_MAX = 0.45
URBAN_NDBI_MIN = 0.02
RURAL_NDVI_MIN = 0.60
RURAL_NDBI_MAX = -0.10


def mask_landsat_qa_pixel(image):
    qa = image.select("QA_PIXEL")
    clear_mask = (
        qa.bitwiseAnd(1 << 0).eq(0)
        .And(qa.bitwiseAnd(1 << 1).eq(0))
        .And(qa.bitwiseAnd(1 << 2).eq(0))
        .And(qa.bitwiseAnd(1 << 3).eq(0))
        .And(qa.bitwiseAnd(1 << 4).eq(0))
        .And(qa.bitwiseAnd(1 << 5).eq(0))
    )
    return image.updateMask(clear_mask).copyProperties(image, ["system:time_start"])


def apply_landsat_scale_factors(image):
    optical = image.select("SR_B.").multiply(0.0000275).add(-0.2)
    thermal = image.select("ST_B10").multiply(0.00341802).add(149.0).rename("ST_B10")
    return image.addBands(optical, None, True).addBands(thermal, None, True)


def add_lst_celsius(image):
    lst = image.select("ST_B10").subtract(273.15).rename("LST")
    return image.addBands(lst, None, True)


def add_ndvi(image):
    ndvi = image.normalizedDifference(["SR_B5", "SR_B4"]).rename("NDVI")
    return image.addBands(ndvi, None, True)


def add_mndwi(image):
    mndwi = image.normalizedDifference(["SR_B3", "SR_B6"]).rename("MNDWI")
    return image.addBands(mndwi, None, True)


def add_ndbi(image):
    ndbi = image.normalizedDifference(["SR_B6", "SR_B5"]).rename("NDBI")
    return image.addBands(ndbi, None, True)


def prepare_landsat_image(image):
    image = mask_landsat_qa_pixel(image)
    image = apply_landsat_scale_factors(image)
    image = add_lst_celsius(image)
    image = add_ndvi(image)
    image = add_mndwi(image)
    image = add_ndbi(image)
    return image


def build_summer_composite(start_date, end_date, label):
    collection = (
        ee.ImageCollection(LANDSAT_COLLECTION)
        .filterBounds(AOI_GEOM)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.eq("PROCESSING_LEVEL", "L2SP"))
        .filter(ee.Filter.lte("CLOUD_COVER", MAX_CLOUD_COVER))
        .map(prepare_landsat_image)
    )
    image_count = collection.size().getInfo()
    print(f"{label}: {image_count} Landsat 8 scenes")
    if image_count == 0:
        raise ValueError(f"No Landsat 8 scenes found for {label}.")
    return collection.median().clip(AOI_GEOM)


## 6. Summer 2015 composite

Build the 2015 summer Landsat 8 composite, print the number of images used, and display an RGB preview.


In [ ]:
image2015 = build_summer_composite(SUMMER_2015_START, SUMMER_2015_END, "Summer 2015")

rgb_vis = {
    "bands": ["SR_B4", "SR_B3", "SR_B2"],
    "min": 0.02,
    "max": 0.30,
    "gamma": 1.2,
}

Map = geemap.Map()
Map.centerObject(AOI_FC, 11)
Map.addLayer(image2015, rgb_vis, "RGB 2015")
Map.addLayer(AOI_FC, {"color": "red"}, "Study area", False)
Map


## 7. Summer 2026 composite

Build the 2026 summer Landsat 8 composite, print the number of images used, and display an RGB preview.


In [ ]:
image2026 = build_summer_composite(SUMMER_2026_START, SUMMER_2026_END, "Summer 2026")

Map = geemap.Map()
Map.centerObject(AOI_FC, 11)
Map.addLayer(image2026, rgb_vis, "RGB 2026")
Map.addLayer(AOI_FC, {"color": "red"}, "Study area", False)
Map


## 8. Spectral indices

Confirm that both composites contain LST, NDVI, MNDWI, and NDBI without duplicated index-band names.


In [ ]:
REQUIRED_ANALYSIS_BANDS = ["LST", "NDVI", "MNDWI", "NDBI"]


def validate_analysis_bands(image, label):
    band_names = image.bandNames().getInfo()
    missing = [band for band in REQUIRED_ANALYSIS_BANDS if band not in band_names]
    duplicated_index_names = [band for band in band_names if band in ["NDVI_1", "MNDWI_1", "NDBI_1", "LST_1"]]
    if missing:
        raise ValueError(f"{label} is missing required bands: {missing}")
    if duplicated_index_names:
        raise ValueError(f"{label} contains duplicated index bands: {duplicated_index_names}")
    print(f"{label} required bands present: {', '.join(REQUIRED_ANALYSIS_BANDS)}")


validate_analysis_bands(image2015, "2015 composite")
validate_analysis_bands(image2026, "2026 composite")

index_vis = {
    "min": -0.5,
    "max": 0.8,
    "palette": ["#7f3b08", "#f6e8c3", "#c7eae5", "#01665e"],
}

Map = geemap.Map()
Map.centerObject(AOI_FC, 11)
Map.addLayer(image2015.select("NDVI"), index_vis, "NDVI 2015", False)
Map.addLayer(image2026.select("NDVI"), index_vis, "NDVI 2026", False)
Map.addLayer(image2015.select("NDBI"), {"min": -0.4, "max": 0.4, "palette": ["#2c7bb6", "#ffffbf", "#d7191c"]}, "NDBI 2015", False)
Map.addLayer(image2026.select("NDBI"), {"min": -0.4, "max": 0.4, "palette": ["#2c7bb6", "#ffffbf", "#d7191c"]}, "NDBI 2026", False)
Map.addLayer(AOI_FC, {"color": "black"}, "Study area", False)
Map


## 9. Urban mask

The urban mask follows the existing spectral-index approach. It excludes water using MNDWI, dense vegetation using NDVI, and non-built-up surfaces using NDBI. A common urban mask is used so 2015 and 2026 are compared over the same built-up-like pixels.


In [ ]:
def build_urban_mask(image):
    return (
        image.select("MNDWI").lt(0.0)
        .And(image.select("NDVI").lt(URBAN_NDVI_MAX))
        .And(image.select("NDBI").gt(URBAN_NDBI_MIN))
    )


urban_mask2015 = build_urban_mask(image2015)
urban_mask2026 = build_urban_mask(image2026)
common_urban_mask = urban_mask2015.And(urban_mask2026).rename("Urban_Mask")

urban_lst2015 = image2015.select("LST").updateMask(common_urban_mask).rename("Urban_LST_2015")
urban_lst2026 = image2026.select("LST").updateMask(common_urban_mask).rename("Urban_LST_2026")
delta_lst = urban_lst2026.subtract(urban_lst2015).rename("Delta_LST")

lst_vis = {
    "min": 18,
    "max": 42,
    "palette": ["#313695", "#4575b4", "#74add1", "#abd9e9", "#ffffbf", "#fdae61", "#f46d43", "#d73027", "#a50026"],
}

delta_lst_vis = {
    "min": -6,
    "max": 6,
    "palette": ["#2166ac", "#67a9cf", "#d1e5f0", "#f7f7f7", "#fddbc7", "#ef8a62", "#b2182b"],
}

Map = geemap.Map()
Map.centerObject(AOI_FC, 11)
Map.addLayer(common_urban_mask.selfMask(), {"palette": ["#fdae61"]}, "Common urban mask", True)
Map.addLayer(urban_lst2015, lst_vis, "Urban LST 2015", False)
Map.addLayer(urban_lst2026, lst_vis, "Urban LST 2026", False)
Map.addLayer(delta_lst, delta_lst_vis, "Delta urban LST 2026 - 2015", False)
Map.addLayer(AOI_FC, {"color": "black"}, "Study area", False)
Map


## 10. Rural reference

The rural reference mask uses vegetated, non-water, non-built-up pixels inside the study area. The mean rural LST is calculated independently for 2015 and 2026.


In [ ]:
def build_rural_mask(image):
    return (
        image.select("NDVI").gt(RURAL_NDVI_MIN)
        .And(image.select("NDBI").lt(RURAL_NDBI_MAX))
        .And(image.select("MNDWI").lt(0.0))
    )


def reduce_mean(image, band_name, geometry=AOI_GEOM):
    result = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=geometry,
        scale=EXPORT_SCALE,
        maxPixels=1e13,
        bestEffort=True,
    )
    return ee.Number(result.get(band_name))


rural_mask2015 = build_rural_mask(image2015)
rural_mask2026 = build_rural_mask(image2026)

rural_lst2015 = image2015.select("LST").updateMask(rural_mask2015).rename("Rural_LST_2015")
rural_lst2026 = image2026.select("LST").updateMask(rural_mask2026).rename("Rural_LST_2026")

rural_mean2015 = reduce_mean(rural_lst2015, "Rural_LST_2015")
rural_mean2026 = reduce_mean(rural_lst2026, "Rural_LST_2026")

print("Mean rural LST 2015 (deg C):", rural_mean2015.getInfo())
print("Mean rural LST 2026 (deg C):", rural_mean2026.getInfo())

Map = geemap.Map()
Map.centerObject(AOI_FC, 11)
Map.addLayer(rural_mask2015.selfMask(), {"palette": ["#1a9850"]}, "Rural reference pixels 2015", False)
Map.addLayer(rural_mask2026.selfMask(), {"palette": ["#006837"]}, "Rural reference pixels 2026", True)
Map.addLayer(AOI_FC, {"color": "black"}, "Study area", False)
Map


## 11. Surface Urban Heat Island intensity

SUHI is calculated as:

```text
SUHI_2015 = Urban_LST_2015 - Mean_Rural_LST_2015
SUHI_2026 = Urban_LST_2026 - Mean_Rural_LST_2026
```


In [ ]:
suhi2015 = urban_lst2015.subtract(rural_mean2015).rename("SUHI_2015")
suhi2026 = urban_lst2026.subtract(rural_mean2026).rename("SUHI_2026")

suhi_vis = {
    "min": -2,
    "max": 8,
    "palette": ["#313695", "#74add1", "#ffffbf", "#f46d43", "#a50026"],
}

Map = geemap.Map()
Map.centerObject(AOI_FC, 11)
Map.addLayer(suhi2015, suhi_vis, "SUHI 2015", True)
Map.addLayer(suhi2026, suhi_vis, "SUHI 2026", False)
Map.addLayer(AOI_FC, {"color": "black"}, "Study area", False)
Map


## 12. Delta SUHI

Delta SUHI is calculated as:

```text
Delta_SUHI = SUHI_2026 - SUHI_2015
```

Positive values mean increased relative urban heat island intensity. Negative values mean decreased relative urban heat island intensity. This is land surface temperature, not air temperature.


In [ ]:
delta_suhi = suhi2026.subtract(suhi2015).rename("Delta_SUHI")

delta_suhi_vis = {
    "min": -4,
    "max": 4,
    "palette": ["#2166ac", "#67a9cf", "#d1e5f0", "#f7f7f7", "#fddbc7", "#ef8a62", "#b2182b"],
}

Map = geemap.Map()
Map.centerObject(AOI_FC, 11)
Map.addLayer(delta_suhi, delta_suhi_vis, "Delta SUHI 2026 - 2015", True)
Map.addLayer(AOI_FC, {"color": "black"}, "Study area", False)
Map


## 13. Summary statistics

Calculate mean, minimum, maximum, and standard deviation for urban LST, rural reference LST, SUHI, and change layers. The table is exported to `outputs/tables/summary_statistics.csv`.


In [ ]:
def summary_stats(image, band_name, label):
    reducer = (
        ee.Reducer.minMax()
        .combine(ee.Reducer.mean(), sharedInputs=True)
        .combine(ee.Reducer.stdDev(), sharedInputs=True)
    )
    result = image.reduceRegion(
        reducer=reducer,
        geometry=AOI_GEOM,
        scale=EXPORT_SCALE,
        maxPixels=1e13,
        bestEffort=True,
    ).getInfo()
    return {
        "layer": label,
        "mean": result.get(f"{band_name}_mean"),
        "min": result.get(f"{band_name}_min"),
        "max": result.get(f"{band_name}_max"),
        "standard_deviation": result.get(f"{band_name}_stdDev"),
    }


stats = [
    summary_stats(urban_lst2015, "Urban_LST_2015", "Urban LST 2015"),
    summary_stats(urban_lst2026, "Urban_LST_2026", "Urban LST 2026"),
    summary_stats(delta_lst, "Delta_LST", "Delta LST"),
    summary_stats(rural_lst2015, "Rural_LST_2015", "Mean rural LST 2015"),
    summary_stats(rural_lst2026, "Rural_LST_2026", "Mean rural LST 2026"),
    summary_stats(suhi2015, "SUHI_2015", "SUHI 2015"),
    summary_stats(suhi2026, "SUHI_2026", "SUHI 2026"),
    summary_stats(delta_suhi, "Delta_SUHI", "Delta SUHI"),
]

summary_df = pd.DataFrame(stats)
summary_path = TABLE_DIR / "summary_statistics.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Saved: {summary_path}")
summary_df


## 14. Export GeoTIFFs

Export the main raster layers as local GeoTIFFs at 30 m resolution, clipped to the study area.


In [ ]:
def export_geotiff(image, filename):
    output_path = GEOTIFF_DIR / filename
    geemap.ee_export_image(
        image,
        filename=str(output_path),
        scale=EXPORT_SCALE,
        region=AOI_GEOM,
        file_per_band=False,
    )
    print(f"Saved: {output_path}")


geotiff_exports = {
    "LST_2015.tif": urban_lst2015,
    "LST_2026.tif": urban_lst2026,
    "Delta_LST.tif": delta_lst,
    "SUHI_2015.tif": suhi2015,
    "SUHI_2026.tif": suhi2026,
    "Delta_SUHI.tif": delta_suhi,
}

for filename, image in geotiff_exports.items():
    export_geotiff(image, filename)


## 15. Export publication-style figures

Create PNG figures suitable for GitHub and LinkedIn. Figures include clear titles, colorbars where relevant, and data source notes.


In [ ]:
def add_source_note(fig, y=0.02):
    fig.text(
        0.02,
        y,
        "Source: Landsat 8 Collection 2 Level 2 / Google Earth Engine; IDE Chile DPA 2023. LST is land surface temperature, not air temperature.",
        fontsize=8,
        ha="left",
    )


def save_study_area_figure():
    fig, ax = plt.subplots(figsize=(8, 8))
    aoi_gdf.plot(ax=ax, color="#d8edf3", edgecolor="#1f2d3a", linewidth=1.2)
    ax.set_title("Study Area: Greater Concepción, Chile", fontsize=15, fontweight="bold", pad=12)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(True, linewidth=0.3, alpha=0.5)
    ax.text(
        0.02,
        0.03,
        "Municipalities: Concepción, Talcahuano, Hualpén, San Pedro de la Paz",
        transform=ax.transAxes,
        fontsize=8,
        ha="left",
        va="bottom",
    )
    add_source_note(fig)
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    output_path = FIGURE_DIR / "Figure_01_Study_Area.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved: {output_path}")


def read_raster(path):
    with rasterio.open(path) as src:
        array = src.read(1).astype("float32")
        if src.nodata is not None:
            array[array == src.nodata] = np.nan
        array[array < -9990] = np.nan
        bounds = src.bounds
        extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
    return array, extent


def percentile_limits(array, low=2, high=98):
    values = array[np.isfinite(array)]
    if values.size == 0:
        raise ValueError("Raster contains no finite values.")
    return np.percentile(values, [low, high])


def symmetric_limits(array, percentile=98):
    values = array[np.isfinite(array)]
    if values.size == 0:
        raise ValueError("Raster contains no finite values.")
    limit = np.percentile(np.abs(values), percentile)
    if limit == 0:
        limit = 1
    return -limit, limit


def save_raster_figure(input_file, output_file, title, cmap, colorbar_label, diverging=False, fixed_limits=None):
    raster_path = GEOTIFF_DIR / input_file
    if not raster_path.exists():
        raise FileNotFoundError(f"Missing raster: {raster_path}")

    array, extent = read_raster(raster_path)

    if fixed_limits is not None:
        vmin, vmax = fixed_limits
    elif diverging:
        vmin, vmax = symmetric_limits(array)
    else:
        vmin, vmax = percentile_limits(array)

    fig, ax = plt.subplots(figsize=(9, 8))

    if diverging:
        norm = TwoSlopeNorm(vcenter=0, vmin=vmin, vmax=vmax)
        image = ax.imshow(array, extent=extent, cmap=cmap, norm=norm)
    else:
        image = ax.imshow(array, extent=extent, cmap=cmap, vmin=vmin, vmax=vmax)

    ax.set_title(title, fontsize=15, fontweight="bold", pad=12)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(False)

    colorbar = plt.colorbar(image, ax=ax, shrink=0.75)
    colorbar.set_label(colorbar_label)

    add_source_note(fig)
    fig.text(0.02, 0.00, "Demonstration workflow; not an operational climate-risk product.", fontsize=8, ha="left")
    plt.tight_layout(rect=[0, 0.06, 1, 1])
    output_path = FIGURE_DIR / output_file
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved: {output_path}")


def save_comparison_figure():
    files = ["LST_2015.tif", "LST_2026.tif", "Delta_SUHI.tif"]
    titles = ["Urban LST 2015", "Urban LST 2026", "Delta SUHI 2026 - 2015"]
    cmaps = ["inferno", "inferno", "RdBu_r"]
    labels = ["LST (deg C)", "LST (deg C)", "Delta SUHI (deg C)"]
    rasters = [read_raster(GEOTIFF_DIR / file) for file in files]

    lst_values = np.concatenate([rasters[0][0].ravel(), rasters[1][0].ravel()])
    lst_values = lst_values[np.isfinite(lst_values)]
    lst_vmin, lst_vmax = np.percentile(lst_values, [2, 98])
    delta_vmin, delta_vmax = symmetric_limits(rasters[2][0])

    fig, axes = plt.subplots(1, 3, figsize=(18, 7))
    for index, ax in enumerate(axes):
        array, extent = rasters[index]
        if index < 2:
            image = ax.imshow(array, extent=extent, cmap=cmaps[index], vmin=lst_vmin, vmax=lst_vmax)
        else:
            norm = TwoSlopeNorm(vcenter=0, vmin=delta_vmin, vmax=delta_vmax)
            image = ax.imshow(array, extent=extent, cmap=cmaps[index], norm=norm)
        ax.set_title(titles[index], fontsize=13, fontweight="bold")
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        colorbar = plt.colorbar(image, ax=ax, shrink=0.7)
        colorbar.set_label(labels[index])

    fig.suptitle("Surface Urban Heat Island Change in Greater Concepción, Chile", fontsize=17, fontweight="bold")
    add_source_note(fig)
    fig.text(0.02, 0.00, "Demonstration workflow; not an operational climate-risk product.", fontsize=8, ha="left")
    plt.tight_layout(rect=[0, 0.06, 1, 0.94])
    output_path = FIGURE_DIR / "Figure_08_Comparison_LST_SUHI.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved: {output_path}")


save_study_area_figure()
save_raster_figure("LST_2015.tif", "Figure_02_LST_2015.png", "Urban Land Surface Temperature - 2015", "inferno", "LST (deg C)", fixed_limits=(18, 42))
save_raster_figure("LST_2026.tif", "Figure_03_LST_2026.png", "Urban Land Surface Temperature - 2026", "inferno", "LST (deg C)", fixed_limits=(18, 42))
save_raster_figure("Delta_LST.tif", "Figure_04_Delta_LST.png", "Urban LST Change: 2026 - 2015", "RdBu_r", "Delta LST (deg C)", diverging=True)
save_raster_figure("SUHI_2015.tif", "Figure_05_SUHI_2015.png", "Surface Urban Heat Island Intensity - 2015", "RdYlBu_r", "SUHI (deg C)", fixed_limits=(-2, 8))
save_raster_figure("SUHI_2026.tif", "Figure_06_SUHI_2026.png", "Surface Urban Heat Island Intensity - 2026", "RdYlBu_r", "SUHI (deg C)", fixed_limits=(-2, 8))
save_raster_figure("Delta_SUHI.tif", "Figure_07_Delta_SUHI.png", "SUHI Change: 2026 - 2015", "RdBu_r", "Delta SUHI (deg C)", diverging=True)
save_comparison_figure()


## 16. Interpretation and limitations

The final **Delta SUHI** layer shows where relative urban heat island intensity increased or decreased between summer 2015 and summer 2026.

- Positive Delta SUHI values indicate increased relative urban heat island intensity.
- Negative Delta SUHI values indicate decreased relative urban heat island intensity.
- LST is land surface temperature, not air temperature.
- SUHI is relative to the rural reference defined in this workflow.
- Results depend on cloud masking, rural reference definition, spectral thresholds, Landsat 30 m resolution, coastal influence, and season selection.
- This is a demonstration workflow, not an operational climate-risk product.
